In [ ]:
from matplotlib import pyplot as plt
import numpy as np
import json
from pprint import pprint
import os

In [ ]:
#--- Plot Params ---
FIGSIZE = (10, 10)
FONTSIZE = 13

filepath = "./board_output/"
plot_dir_path = filepath +"paper_plots/"
os.makedirs(plot_dir_path, exist_ok=True)


plt.rc('font', size=FONTSIZE)      # default text
# plt.rc('axes', titlesize=16) # title
plt.rc('axes', labelsize=15) # x/y labels
# plt.rc('xtick', labelsize=12) # tick labels
# plt.rc('ytick', labelsize=12)
# plt.rc('legend', fontsize=12)

In [ ]:
def to_signed_int16(n):
    n = n & 0xFFFF
    return n - 0x10000 if n & 0x8000 else n

def cleanData(filename, is_signed=True):
    data = np.loadtxt(filename, delimiter=',', dtype=np.uint16)
    if is_signed:
        # Reinterpret uint16 as int16 (much faster than vectorize)
        return data.view(np.int16)
    else:
        return data

def getAllRunsData(filenames, full_run_path, is_signed=True):
    all_data = []
    for filename in filenames:
        all_data.append(cleanData(full_run_path + filename, is_signed=is_signed))
    return all_data

In [ ]:
def get_filepaths(run_name):
    full_run_path = filepath + run_name + "/"
    paths = {}

    all_paths = os.listdir(full_run_path)
    paths["att_fit_paths"] = sorted([x for x in all_paths if "attacker_fits" in x])
    paths["att_mutrate_paths"] = sorted([x for x in all_paths if "attacker_mutRates" in x])
    paths["att_prob_large_dist_paths"] = sorted([x for x in all_paths if "attacker_prob_large_dist" in x])
    paths["def_fit_paths"] = sorted([x for x in all_paths if "defender_fits" in x])
    paths["def_mutrate_paths"] = sorted([x for x in all_paths if "defender_mutRates" in x])
    paths["def_prob_large_dist_paths"] = sorted([x for x in all_paths if "defender_prob_large_dist" in x])
    paths["num_runs"] = len(paths["att_fit_paths"])
    paths["run_name"] = run_name

    with open(full_run_path + "run_params.json", "r") as f:
        paths["run_params"] = json.load(f)

    return paths

def get_data(paths, data_name, is_signed=False):
    full_run_path = filepath + paths["run_name"] + "/"
    att_runs = getAllRunsData(paths[f"att_{data_name}_paths"], full_run_path, is_signed)
    def_runs = getAllRunsData(paths[f"def_{data_name}_paths"], full_run_path, is_signed)
    att_run_medians = np.median(att_runs, axis=2)
    def_run_medians = np.median(def_runs, axis=2)
    att_mean_medians = np.mean(att_run_medians, axis=0)
    def_mean_medians = np.mean(def_run_medians, axis=0)
    att_median_std = np.std(att_run_medians, axis=0)
    def_median_std = np.std(def_run_medians, axis=0)

    data = {
        "att_med" : att_mean_medians,
        "def_med" : def_mean_medians,
        "att_std" : att_median_std,
        "def_std" : def_median_std
    }

    return data

In [ ]:
# Define a function to plot on a given axis
def plot_on_ax(ax, x_axis, def_mean_medians, def_median_std, att_mean_medians, att_median_std, run_params, y_label, title, budgets, boff=0, show_legend=True, showx=True):
    num_budget = len(budgets)
    ax.plot(x_axis, def_mean_medians, linewidth=2, label='Defender')
    ax.fill_between(x_axis, def_mean_medians - def_median_std, def_mean_medians + def_median_std, alpha=0.3, label='_nolegend_')
    ax.plot(x_axis, att_mean_medians, linewidth=2, label='Attacker')
    ax.fill_between(x_axis, att_mean_medians - att_median_std, att_mean_medians + att_median_std, alpha=0.3, label='_nolegend_')
    
    budget_gens = [(run_params["generations"]//num_budget)*i for i in range(0, num_budget)]
    for i in range(1, len(budget_gens)):
        ax.axvline(x=budget_gens[i], color="black", linestyle='--')

    trans = ax.get_xaxis_transform()
    for i, bgen in enumerate(budget_gens):
        ax.text(bgen + budget_gens[1] // 2, 0.95 +boff, f'{budgets[i]}', transform=trans, ha='center', va='bottom', fontsize=FONTSIZE, fontweight='bold')
    
    ax.set_xlabel('Generation')
    if not showx:
        ax.set_xticks([])
    ax.set_ylabel(f'Med. {y_label}')
    ax.set_title(title)
    if show_legend:
        ax.legend()
    ax.grid(True, alpha=0.3)

def plot_with_bottom_bars(
    fig, gs1, gs2,
    x_axis,
    def_mean_medians, def_median_std,
    att_mean_medians, att_median_std,
    sec_values1,
    sec_values2,
    run_params, y_label, sec_label,
    title, budgets, boff=0,
    show_legend=True,
    showx=True
):
    # Top axis: your existing line plot
    ax_top = fig.add_subplot(gs1)
    plot_on_ax(
        ax_top,
        x_axis,
        def_mean_medians, def_median_std,
        att_mean_medians, att_median_std,
        run_params, y_label, title,
        budgets, boff=boff,
        show_legend=show_legend,
        showx = False
    )

    width = 3
    x_axis = np.asarray(x_axis)

    # Bottom axis: bars, share x with top
    ax_bottom = fig.add_subplot(gs2, sharex=ax_top)
    ax_bottom.bar(x_axis - width/2, sec_values1, width=width,
                  color='tab:blue', alpha=0.7)
    ax_bottom.bar(x_axis + width/2, sec_values2, width=width,
                  color='tab:orange', alpha=0.7)
    ax_bottom.set_ylabel(sec_label)
    if showx:
        ax_bottom.set_xlabel('Generation')
    else:
        ax_bottom.set_xticks([])
    ax_bottom.grid(True, alpha=0.3)
    ax_bottom.set_yscale('log')
    ax_bottom.set_yticks([1, 10, 100, 1000])

    num_budget = len(budgets)
    budget_gens = [(run_params["generations"]//num_budget)*i for i in range(0, num_budget)]
    for i in range(1, len(budget_gens)):
        ax_bottom.axvline(x=budget_gens[i], color="black", linestyle='--')

    # Optional: hide x tick labels on top
    # plt.setp(ax_top.get_xticklabels(), visible=False)


In [ ]:

# Define a function to plot on a given axis
def plot_on_ax(ax, x_axis, def_mean_medians, def_median_std, att_mean_medians, att_median_std, run_params, y_label, title, budgets, boff=0, show_legend=True, showx=True):
    num_budget = len(budgets)
    ax.plot(x_axis, def_mean_medians, linewidth=2, label='Defender')
    ax.fill_between(x_axis, def_mean_medians - def_median_std, def_mean_medians + def_median_std, alpha=0.3, label='_nolegend_')
    ax.plot(x_axis, att_mean_medians, linewidth=2, label='Attacker')
    ax.fill_between(x_axis, att_mean_medians - att_median_std, att_mean_medians + att_median_std, alpha=0.3, label='_nolegend_')
    
    budget_gens = [(run_params["generations"]//num_budget)*i for i in range(0, num_budget)]
    for i in range(1, len(budget_gens)):
        ax.axvline(x=budget_gens[i], color="black", linestyle='--')

    trans = ax.get_xaxis_transform()
    for i, bgen in enumerate(budget_gens):
        ax.text(bgen + budget_gens[1] // 2, 0.95 +boff, f'{budgets[i]}', transform=trans, ha='center', va='bottom', fontsize=FONTSIZE, fontweight='bold')
    
    ax.set_xlabel('Generation')
    if not showx:
        ax.set_xticks([])
    ax.set_ylabel(f'Med. {y_label}')
    ax.set_title(title)
    if show_legend:
        ax.legend()
    ax.grid(True, alpha=0.3)

In [ ]:
def plot_with_bottom_bars(
    fig, gs1, gs2,
    x_axis,
    def_mean_medians, def_median_std,
    att_mean_medians, att_median_std,
    sec_values1,
    sec_values2,
    run_params, y_label, sec_label,
    title, budgets, boff=0,
    show_legend=True,
    showx=True
):
    # Top axis: your existing line plot
    ax_top = fig.add_subplot(gs1)
    plot_on_ax(
        ax_top,
        x_axis,
        def_mean_medians, def_median_std,
        att_mean_medians, att_median_std,
        run_params, y_label, title,
        budgets, boff=boff,
        show_legend=show_legend,
        showx = False
    )

    width = 3
    x_axis = np.asarray(x_axis)

    # Bottom axis: bars, share x with top
    ax_bottom = fig.add_subplot(gs2, sharex=ax_top)
    ax_bottom.bar(x_axis - width/2, sec_values1, width=width,
                  color='tab:blue', alpha=0.7)
    ax_bottom.bar(x_axis + width/2, sec_values2, width=width,
                  color='tab:orange', alpha=0.7)
    ax_bottom.set_ylabel(sec_label)
    if showx:
        ax_bottom.set_xlabel('Generation')
    else:
        ax_bottom.set_xticks([])
    ax_bottom.grid(True, alpha=0.3)
    ax_bottom.set_yscale('log')
    ax_bottom.set_yticks([1, 10, 100, 1000])

    num_budget = len(budgets)
    budget_gens = [(run_params["generations"]//num_budget)*i for i in range(0, num_budget)]
    for i in range(1, len(budget_gens)):
        ax_bottom.axvline(x=budget_gens[i], color="black", linestyle='--')

    # Optional: hide x tick labels on top
    # plt.setp(ax_top.get_xticklabels(), visible=False)


In [ ]:
def plot_with_mut_rate_panel(
    fig, gs,
    x_axis,
    def_mean_medians, def_median_std,
    att_mean_medians, att_median_std,
    def_mut_rate, att_mut_rate,          # NEW
    run_params, y_label,
    title, budgets, boff=0,
    show_legend=True
):
    # Top: payoffs (reuse your existing function)
    ax_top = fig.add_subplot(gs[0])
    plot_on_ax(
        ax_top,
        x_axis,
        def_mean_medians, def_median_std,
        att_mean_medians, att_median_std,
        run_params, y_label, title,
        budgets, boff=boff,
        show_legend=show_legend
    )

    # Bottom: mutation rates, share x with top
    ax_bottom = fig.add_subplot(gs[1], sharex=ax_top)

    # Option A: lines
    ax_bottom.plot(x_axis, def_mut_rate, color='tab:blue',
                   linewidth=2, label='Defender mutation rate')
    ax_bottom.plot(x_axis, att_mut_rate, color='tab:orange',
                   linewidth=2, label='Attacker mutation rate')

    # Option B (alternative): bars (stacked or side-by-side)
    # width = 0.4
    # ax_bottom.bar(x_axis - width/2, def_mut_rate, width=width,
    #               color='tab:blue', alpha=0.7, label='Defender mutation rate')
    # ax_bottom.bar(x_axis + width/2, att_mut_rate, width=width,
    #               color='tab:red', alpha=0.7, label='Attacker mutation rate')

    ax_bottom.set_ylabel('Med. Threshold')
    ax_bottom.set_xlabel('Generation')
    ax_bottom.grid(True, alpha=0.3)
    # ax_bottom.legend()

    ax_bottom.set_yscale('log')
    ax_bottom.set_yticks([1, 10, 100, 1000])

    # Hide x labels on top axis to avoid clutter
    plt.setp(ax_top.get_xticklabels(), visible=False)

In [ ]:
def plot_with_bottom_lines(
    fig, gs1, gs2,
    x_axis,
    def_mean_medians, def_median_std,
    att_mean_medians, att_median_std,
    sec_values1,
    sec_values2,
    run_params, y_label, sec_label,
    title, budgets, boff=0,
    show_legend=True,
    showx=True
):
    # Top axis: your existing line plot
    ax_top = fig.add_subplot(gs1)
    plot_on_ax(
        ax_top,
        x_axis,
        def_mean_medians, def_median_std,
        att_mean_medians, att_median_std,
        run_params, y_label, title,
        budgets, boff=boff,
        show_legend=show_legend,
        showx = showx
    )

    x_axis = np.asarray(x_axis)

    # Bottom axis: lines, share x with top
    ax_bottom = fig.add_subplot(gs2, sharex=ax_top)
    ax_bottom.plot(x_axis, sec_values1, color='tab:blue',
                linewidth=2, label='Defender mutation rate')
    ax_bottom.plot(x_axis, sec_values2, color='tab:orange',
                linewidth=2, label='Attacker mutation rate')
    ax_bottom.set_ylabel(sec_label)

    if showx:
        ax_bottom.set_xlabel('Generation')
        # FORCE tick labels to show despite sharex
        plt.setp(ax_bottom.get_xticklabels(), visible=True)
    else:
        ax_bottom.set_xticks([])
    ax_bottom.grid(True, alpha=0.3)
    ax_bottom.set_yscale('log')
    ax_bottom.set_yticks([1, 10, 100, 1000])

    num_budget = len(budgets)
    budget_gens = [(run_params["generations"]//num_budget)*i for i in range(0, num_budget)]
    for i in range(1, len(budget_gens)):
        ax_bottom.axvline(x=budget_gens[i], color="black", linestyle='--')

    # Optional: hide x tick labels on top
    # plt.setp(ax_top.get_xticklabels(), visible=False)


In [ ]:
run_name = "AA_default"
paths = get_filepaths(run_name)
aa_fit_data = get_data(paths, "fit", is_signed=True)
aa_mutrate_data = get_data(paths, "mutrate")
aa_pd_data = get_data(paths, "prob_large_dist")

In [ ]:

run_name = "SA_default"
paths = get_filepaths(run_name)
sa_fit_data = get_data(paths, "fit", is_signed=True)
sa_mutrate_data = get_data(paths, "mutrate")
sa_pd_data = get_data(paths, "prob_large_dist")


In [ ]:
run_name = "AS_default"
paths = get_filepaths(run_name)
as_fit_data = get_data(paths, "fit", is_signed=True)
as_mutrate_data = get_data(paths, "mutrate")
as_pd_data = get_data(paths, "prob_large_dist")

In [ ]:
run_name = "SS_default"
paths = get_filepaths(run_name)
ss_fit_data = get_data(paths, "fit", is_signed=True)
ss_mutrate_data = get_data(paths, "mutrate")
ss_pd_data = get_data(paths, "prob_large_dist")

In [ ]:
# # fig, axs = plt.subplots(1, 1, figsize=(10,6), sharex=True, sharey=False)
# fig = plt.figure(figsize=(8, 6))
# gs = fig.add_gridspec(2, 1, height_ratios=[3, 1], hspace=0.05)
# budgets = [28, 280, 2800]

# # Plot on each subplot - replace with your four data sets
# run_params = paths["run_params"]
# x_axis = [x for x in range(0, run_params["generations"] + run_params["checkpoint_interval"], run_params["checkpoint_interval"])]


# plot_with_mut_rate_panel(
#     fig, gs,
#     x_axis,
#     aa_fit_data["def_med"], aa_fit_data["def_std"],
#     aa_fit_data["att_med"], aa_fit_data["att_std"],
#     aa_mutrate_data["def_med"], aa_mutrate_data["att_med"],
#     run_params, y_label="Payoff",
#     title="Payoff and mutation rates",
#     budgets=budgets,
#     boff=-0.03
# )


In [ ]:
# fig = plt.figure(figsize=(8, 6))
# gs = fig.add_gridspec(2, 1, height_ratios=[3, 1], hspace=0.05)
# run_params = paths["run_params"]
# budgets = [28, 280, 2800]

# x_axis = [x for x in range(0, run_params["generations"] + run_params["checkpoint_interval"], run_params["checkpoint_interval"])]


# plot_with_bottom_bars(
#     fig, gs,
#     x_axis,
#     aa_fit_data["def_med"], aa_fit_data["def_std"],
#     aa_fit_data["att_med"], aa_fit_data["att_std"],
#     sec_values1=aa_mutrate_data["def_med"],
#     sec_values2=aa_mutrate_data["att_med"],
#     run_params=run_params,
#     y_label="Fitness",
#     sec_label="Median Threshold",
#     title="",
#     budgets=budgets,
#     boff= -0.03
# )

In [ ]:

# fig, axs = plt.subplots(1, 1, figsize=(10,6), sharex=True, sharey=False)
# budgets = [28, 280, 2800]

# # Plot on each subplot - replace with your four data sets
# run_params = paths["run_params"]
# x_axis = [x for x in range(0, run_params["generations"] + run_params["checkpoint_interval"], run_params["checkpoint_interval"])]
# plot_on_ax(axs, x_axis, aa_fit_data["def_med"], aa_fit_data["def_std"], aa_fit_data["att_med"], aa_fit_data["att_std"], run_params, "Payoff", "", budgets)
# # plot_on_ax(axs[1], x_axis, aa_mutrate_data["def_med"], aa_mutrate_data["def_std"], aa_mutrate_data["att_med"], aa_mutrate_data["att_std"], run_params, "Mutation Threshold", "Adapt-Threshold Adapt-SDT", budgets)
# # axs[1].set_yscale('log')
# # axs[1].set_yticks([1, 10, 100, 1000])

# plt.tight_layout()
# plt.savefig(plot_dir_path + "s1_aa_payoff.png")
# plt.show()

In [ ]:
# fig, axs = plt.subplots(1, 1, figsize=(10,6), sharex=True, sharey=False)
# budgets = [28, 280, 2800]

# # Plot on each subplot - replace with your four data sets
# run_params = paths["run_params"]
# x_axis = [x for x in range(0, run_params["generations"] + run_params["checkpoint_interval"], run_params["checkpoint_interval"])]
# plot_on_ax(axs, x_axis, aa_mutrate_data["def_med"], aa_mutrate_data["def_std"], aa_mutrate_data["att_med"], aa_mutrate_data["att_std"], run_params, "Payoff", "", budgets)
# # plot_on_ax(axs[1], x_axis, aa_mutrate_data["def_med"], aa_mutrate_data["def_std"], aa_mutrate_data["att_med"], aa_mutrate_data["att_std"], run_params, "Mutation Threshold", "Adapt-Threshold Adapt-SDT", budgets)
# # axs[1].set_yscale('log')
# # axs[1].set_yticks([1, 10, 100, 1000])
# axs.set_yscale('log')
# axs.set_yticks([1, 10, 100, 1000])

# plt.tight_layout()
# plt.savefig(plot_dir_path + "s1_aa_mutrate.png")
# plt.show()

In [ ]:
budgets = [28, 280, 2800]

fig = plt.figure(figsize=(9,13))
gs = fig.add_gridspec(8, 1, height_ratios=[3,0.6, 0.3, 3, 0.8, 0.3, 3, 0.8], hspace=0.04)
run_params = paths["run_params"]
x_axis = [x for x in range(0, run_params["generations"] + run_params["checkpoint_interval"], run_params["checkpoint_interval"])]

# Experiment 1 (rows 0-1, tight)
plot_with_bottom_lines(
    fig, gs[0], gs[1],
    x_axis, ss_fit_data["def_med"], ss_fit_data["def_std"],
    ss_fit_data["att_med"], ss_fit_data["att_std"],
    sec_values1=ss_mutrate_data["def_med"],
    sec_values2=ss_mutrate_data["att_med"],
    run_params=run_params, y_label="Payoff", sec_label="Med. Threshold",
    title="Static-Threshold Static-SDT (S-S)", budgets=budgets, boff=-0.03, showx=False
)

# Spacer row (gs[2] = 0.5 units gap)

# Experiment 2 (rows 3-4, tight)
plot_with_bottom_lines(
    fig, gs[3], gs[4],
    x_axis, as_fit_data["def_med"], as_fit_data["def_std"],
    as_fit_data["att_med"], as_fit_data["att_std"],
    sec_values1=as_mutrate_data["def_med"],
    sec_values2=as_mutrate_data["att_med"],
    run_params=run_params, y_label="Payoff", sec_label="Med. Threshold",
    title="Adapt-Threshold Static-SDT (A-S)", budgets=budgets, boff=-0.03, showx=False, show_legend=False
)

plot_with_bottom_lines(
    fig, gs[6], gs[7],
    x_axis, aa_fit_data["def_med"], aa_fit_data["def_std"],
    aa_fit_data["att_med"], aa_fit_data["att_std"],
    sec_values1=aa_mutrate_data["def_med"],
    sec_values2=aa_mutrate_data["att_med"],
    run_params=run_params, y_label="Payoff", sec_label="Med. Threshold",
    title="Adapt-Threshold Adapt-SDT (A-A)", budgets=budgets, boff=-0.03, show_legend=False
)
# plt.tight_layout()
plt.savefig(plot_dir_path + "payoff_mutrate_subplots.png", bbox_inches='tight')
plt.show()


In [ ]:

# fig, axs = plt.subplots(3, 1, figsize=(10,10), sharex=True, sharey=False)
# budgets = [28, 280, 2800]

# # Plot on each subplot - replace with your four data sets
# run_params = paths["run_params"]
# x_axis = [x for x in range(0, run_params["generations"] + run_params["checkpoint_interval"], run_params["checkpoint_interval"])]
# plot_on_ax(axs[0], x_axis, aa_fit_data["def_med"], aa_fit_data["def_std"], aa_fit_data["att_med"], aa_fit_data["att_std"], run_params, "Payoff", "Adapt-Threshold Adapt-SDT", budgets, boff= -0.05)
# plot_on_ax(axs[1], x_axis, as_fit_data["def_med"], as_fit_data["def_std"], as_fit_data["att_med"], as_fit_data["att_std"], run_params, "Payoff", "Adapt-Threshold Static-SDT", budgets, boff= -0.05, show_legend=False)
# plot_on_ax(axs[2], x_axis, ss_fit_data["def_med"], ss_fit_data["def_std"], ss_fit_data["att_med"], ss_fit_data["att_std"], run_params, "Payoff", "Static-Threshold Static-SDT", budgets, boff= -0.05, show_legend=False)

# plt.tight_layout()
# plt.savefig(plot_dir_path + "med_payoff_subplots.png")
# plt.show()

In [ ]:
# fig, axs = plt.subplots(3, 1, figsize=(10,10), sharex=True, sharey=True)
# budgets = [28, 280, 2800]

# # Plot on each subplot - replace with your four data sets
# run_params = paths["run_params"]

# plot_on_ax(axs[0], x_axis, aa_mutrate_data["def_med"], aa_mutrate_data["def_std"], aa_mutrate_data["att_med"], aa_mutrate_data["att_std"], run_params, "Mut Threshold", "Adapt-Threshold Adapt-SDT", budgets, boff=-0.05)
# axs[0].set_yscale('log')
# axs[0].set_yticks([1, 10, 100, 1000])

# plot_on_ax(axs[1], x_axis, as_mutrate_data["def_med"], as_mutrate_data["def_std"], as_mutrate_data["att_med"], as_mutrate_data["att_std"], run_params, "Mut Threshold", "Adapt-Threshold Static-SDT", budgets, boff=-0.05, show_legend=False)
# axs[1].set_yscale('log')
# axs[1].set_yticks([1, 10, 100, 1000])

# plot_on_ax(axs[2], x_axis, ss_mutrate_data["def_med"], ss_mutrate_data["def_std"], ss_mutrate_data["att_med"], ss_mutrate_data["att_std"], run_params, "Mut Threshold", "Static-Threshold Static-SDT", budgets, boff=-0.05, show_legend=False)
# axs[2].set_yscale('log')
# axs[2].set_yticks([1, 10, 100, 1000])

# plt.tight_layout()
# plt.savefig(plot_dir_path + "med_mutrate_subplots.png")
# plt.show()

In [ ]:
fig, axs = plt.subplots(1, 1, figsize=(10, 5), sharex=True, sharey=False)
budgets = [28, 280, 2800]

# Plot on each subplot - replace with your four data sets
run_params = paths["run_params"]
x_axis = [x for x in range(0, run_params["generations"] + run_params["checkpoint_interval"], run_params["checkpoint_interval"])]
plot_on_ax(axs, x_axis, aa_pd_data["def_med"], aa_pd_data["def_std"], aa_pd_data["att_med"], aa_pd_data["att_std"], run_params, "SDT", "", budgets)

# plt.tight_layout()
plt.savefig(plot_dir_path + "med_sdt_subplots.png", bbox_inches='tight')
plt.show()

In [ ]:

# fig, axs = plt.subplots(3, 2, figsize=FIGSIZE, sharex=True, sharey=False)
# budgets = [28, 280, 2800]

# # Plot on each subplot - replace with your four data sets
# run_params = paths["run_params"]
# x_axis = [x for x in range(0, run_params["generations"] + run_params["checkpoint_interval"], run_params["checkpoint_interval"])]
# plot_on_ax(axs[0, 0], x_axis, aa_fit_data["def_med"], aa_fit_data["def_std"], aa_fit_data["att_med"], aa_fit_data["att_std"], run_params, "Payoff", "Adapt-Threshold Adapt-SDT", budgets)
# # axs[0, 0].set_yticks([0, 1000, 2000, 3000, 4000, 5000])
# plot_on_ax(axs[1, 0], x_axis, as_fit_data["def_med"], as_fit_data["def_std"], as_fit_data["att_med"], as_fit_data["att_std"], run_params, "Payoff", "Adapt-Threshold Static-SDT", budgets)
# plot_on_ax(axs[2, 0], x_axis, ss_fit_data["def_med"], ss_fit_data["def_std"], ss_fit_data["att_med"], ss_fit_data["att_std"], run_params, "Payoff", "Static-Threshold Static-SDT", budgets)

# plot_on_ax(axs[0, 1], x_axis, aa_mutrate_data["def_med"], aa_mutrate_data["def_std"], aa_mutrate_data["att_med"], aa_mutrate_data["att_std"], run_params, "Mutation Threshold", "Adapt-Threshold Adapt-SDT", budgets)
# axs[0,1].set_yscale('log')
# axs[0,1].set_yticks([1, 10, 100, 1000])
# plot_on_ax(axs[1, 1], x_axis, as_mutrate_data["def_med"], as_mutrate_data["def_std"], as_mutrate_data["att_med"], as_mutrate_data["att_std"], run_params, "Mutation Threshold", "Adapt-Threshold Static-SDT", budgets)
# axs[1,1].set_yscale('log')
# axs[1,1].set_yticks([1, 10, 100, 1000])
# plot_on_ax(axs[2, 1], x_axis, ss_mutrate_data["def_med"], ss_mutrate_data["def_std"], ss_mutrate_data["att_med"], ss_mutrate_data["att_std"], run_params, "Mutation Threshold", "Static-Threshold Static-SDT", budgets)
# axs[2,1].set_yscale('log')
# axs[2,1].set_yticks([1, 10, 100, 1000])

# plt.tight_layout()
# plt.savefig(plot_dir_path + "med_subplots.png")
# plt.show()

In [ ]:
run_name = "AA_hill_budget"
paths = get_filepaths(run_name)
hill_fit_data = get_data(paths, "fit", is_signed=True)
hill_mutrate_data = get_data(paths, "mutrate")
hill_pd_data = get_data(paths, "prob_large_dist")

In [ ]:
run_name = "AA_dec_budget"
paths = get_filepaths(run_name)
dec_fit_data = get_data(paths, "fit", is_signed=True)
dec_mutrate_data = get_data(paths, "mutrate")
dec_pd_data = get_data(paths, "prob_large_dist")

In [ ]:
# fig, axs = plt.subplots(3, 1, figsize=FIGSIZE, sharex=True, sharey=False)
# inc_budgets = [28, 280, 2800]
# dec_budgets = [2800, 280, 28]
# hill_budgets = [280, 560, 280]

# # Plot on each subplot - replace with your four data sets
# run_params = paths["run_params"]
# x_axis = [x for x in range(0, run_params["generations"] + run_params["checkpoint_interval"], run_params["checkpoint_interval"])]
# plot_on_ax(axs[0], x_axis, aa_fit_data["def_med"], aa_fit_data["def_std"], aa_fit_data["att_med"], aa_fit_data["att_std"], run_params, "Payoff", "Increasing Budget", inc_budgets, boff=-0.05)
# plot_on_ax(axs[1], x_axis, hill_fit_data["def_med"], hill_fit_data["def_std"], hill_fit_data["att_med"], hill_fit_data["att_std"], run_params, "Payoff", "Hill Budget", hill_budgets, boff=-0.05, show_legend=False)
# plot_on_ax(axs[2], x_axis, dec_fit_data["def_med"], dec_fit_data["def_std"], dec_fit_data["att_med"], dec_fit_data["att_std"], run_params, "Payoff", "Decreasing Budget", dec_budgets, boff=-0.05, show_legend=False)

# plt.tight_layout()
# plt.savefig(plot_dir_path + "budget_payoff_subplots.png")
# plt.show()

In [ ]:
inc_budgets = [28, 280, 2800]
dec_budgets = [2800, 280, 28]
hill_budgets = [280, 560, 280]

fig = plt.figure(figsize=(9,13))
gs = fig.add_gridspec(8, 1, height_ratios=[3,0.6, 0.3, 3, 0.8, 0.3, 3, 0.8], hspace=0.04)
x_axis = [x for x in range(0, run_params["generations"] + run_params["checkpoint_interval"], run_params["checkpoint_interval"])]

# Experiment 1 (rows 0-1, tight)

plot_with_bottom_lines(
    fig, gs[0], gs[1],
    x_axis, aa_fit_data["def_med"], aa_fit_data["def_std"],
    aa_fit_data["att_med"], aa_fit_data["att_std"],
    sec_values1=aa_mutrate_data["def_med"],
    sec_values2=aa_mutrate_data["att_med"],
    run_params=run_params, y_label="Payoff", sec_label="Med Threshold",
    title="Increasing Budget (28, 280, 2800)", budgets=inc_budgets, boff=-0.03, showx=False
)


# Spacer row (gs[2] = 0.5 units gap)

# Experiment 2 (rows 3-4, tight)
plot_with_bottom_lines(
    fig, gs[3], gs[4],
    x_axis, hill_fit_data["def_med"], hill_fit_data["def_std"],
    hill_fit_data["att_med"], hill_fit_data["att_std"],
    sec_values1=hill_mutrate_data["def_med"],
    sec_values2=hill_mutrate_data["att_med"],
    run_params=run_params, y_label="Payoff", sec_label="Med Threshold",
    title="Hill Budget (280, 560, 280)", budgets=hill_budgets, boff=-0.03, showx=False, show_legend=False
)

plot_with_bottom_lines(
    fig, gs[6], gs[7],
    x_axis, dec_fit_data["def_med"], dec_fit_data["def_std"],
    dec_fit_data["att_med"], dec_fit_data["att_std"],
    sec_values1=dec_mutrate_data["def_med"],
    sec_values2=dec_mutrate_data["att_med"],
    run_params=run_params, y_label="Payoff", sec_label="Med Threshold",
    title="Decreasing Budget (2800, 280, 28)", budgets=dec_budgets, boff=-0.03, showx=True, show_legend=False
)


# plt.tight_layout()
plt.savefig(plot_dir_path + "payoff_mutrate_budget_subplots.png", bbox_inches='tight')
plt.show()


In [ ]:
# fig, axs = plt.subplots(3, 1, figsize=FIGSIZE, sharex=True, sharey=False)
# inc_budgets = [28, 280, 2800]
# dec_budgets = [2800, 280, 28]
# hill_budgets = [280, 560, 280]

# # Plot on each subplot - replace with your four data sets
# run_params = paths["run_params"]

# plot_on_ax(axs[0], x_axis, aa_mutrate_data["def_med"], aa_mutrate_data["def_std"], aa_mutrate_data["att_med"], aa_mutrate_data["att_std"], run_params, "Mut Threshold", "Increasing Budget", inc_budgets, boff=-0.05)
# axs[0].set_yscale('log')
# axs[0].set_yticks([1, 10, 100, 1000])

# plot_on_ax(axs[1], x_axis, hill_mutrate_data["def_med"], hill_mutrate_data["def_std"], hill_mutrate_data["att_med"], hill_mutrate_data["att_std"], run_params, "Mut Threshold", "Hill Budget", hill_budgets, boff=-0.05, show_legend=False)
# axs[1].set_yscale('log')
# axs[1].set_yticks([1, 10, 100, 1000])

# plot_on_ax(axs[2], x_axis, dec_mutrate_data["def_med"], dec_mutrate_data["def_std"], dec_mutrate_data["att_med"], dec_mutrate_data["att_std"], run_params, "Mut Threshold", "Decreasing Budget", dec_budgets, boff=-0.05, show_legend=False)
# axs[2].set_yscale('log')
# axs[2].set_yticks([1, 10, 100, 1000])

# plt.tight_layout()
# plt.savefig(plot_dir_path + "budget_mutrate_subplots.png")
# plt.show()

In [ ]:
run_name = "AA_gen-2999_budgets_10"
paths = get_filepaths(run_name)
long_fit_data = get_data(paths, "fit", is_signed=True)
long_mutrate_data = get_data(paths, "mutrate")
long_pd_data = get_data(paths, "prob_large_dist")

In [ ]:

fig, axs = plt.subplots(2, 1, figsize=(12, 8), sharex=True, sharey=False)
budgets = [14*230, 14*80, 14*150, 14*340, 14*90, 14*260, 14*170, 14*40, 14*310, 14*120]

# Plot on each subplot - replace with your four data sets
run_params = paths["run_params"]
# pprint(run_params)
x_axis = [x for x in range(0, run_params["generations"] + run_params["checkpoint_interval"], run_params["checkpoint_interval"])]
plot_on_ax(axs[0], x_axis, long_fit_data["def_med"], long_fit_data["def_std"], long_fit_data["att_med"], long_fit_data["att_std"], run_params, "Payoff", "", budgets, boff=-0.03, show_legend=False)
plot_on_ax(axs[1], x_axis, long_mutrate_data["def_med"], long_mutrate_data["def_std"], long_mutrate_data["att_med"], long_mutrate_data["att_std"], run_params, "Mutation Threshold", "", budgets, boff=-0.03)
axs[1].set_yscale('log')
axs[1].set_yticks([1, 10, 100, 1000, 10000])

# plt.tight_layout()
plt.savefig(plot_dir_path + "longrun_subplots.png", bbox_inches='tight')
plt.show()

In [ ]:
run_name = "AS_fixed-step_1"
paths = get_filepaths(run_name)
fix1_fit_data = get_data(paths, "fit", is_signed=True)
fix1_mutrate_data = get_data(paths, "mutrate")
fix1_pd_data = get_data(paths, "prob_large_dist")

run_name = "AS_fixed-step_6"
paths = get_filepaths(run_name)
fix6_fit_data = get_data(paths, "fit", is_signed=True)
fix6_mutrate_data = get_data(paths, "mutrate")
fix6_pd_data = get_data(paths, "prob_large_dist")

run_name = "AS_fixed-step_65"
paths = get_filepaths(run_name)
fix65_fit_data = get_data(paths, "fit", is_signed=True)
fix65_mutrate_data = get_data(paths, "mutrate")
fix65_pd_data = get_data(paths, "prob_large_dist")

run_name = "AS_fixed-step_655"
paths = get_filepaths(run_name)
fix655_fit_data = get_data(paths, "fit", is_signed=True)
fix655_mutrate_data = get_data(paths, "mutrate")
fix655_pd_data = get_data(paths, "prob_large_dist")

In [ ]:
fig, axs = plt.subplots(2, 1, figsize=FIGSIZE, sharex=True, sharey=False)
budgets = [28, 280, 2800]
num_budget = len(budgets)

# Plot on each subplot - replace with your four data sets
run_params = paths["run_params"]
x_axis = [x for x in range(0, run_params["generations"] + run_params["checkpoint_interval"], run_params["checkpoint_interval"])]

defs_fit_med = [fix1_fit_data["def_med"], fix6_fit_data["def_med"], fix65_fit_data["def_med"], fix655_fit_data["def_med"]]
defs_fit_std = [fix1_fit_data["def_std"], fix6_fit_data["def_std"], fix65_fit_data["def_std"], fix655_fit_data["def_std"]]
atts_fit_med = [fix1_fit_data["att_med"], fix6_fit_data["att_med"], fix65_fit_data["att_med"], fix655_fit_data["att_med"]]
atts_fit_std = [fix1_fit_data["att_std"], fix6_fit_data["att_std"], fix65_fit_data["att_std"], fix655_fit_data["att_std"]]

labels = [1, 6, 65, 655]

for i in range(len(defs_fit_med)):
    axs[0].plot(x_axis, defs_fit_med[i], linewidth=2, label=f'Step Size: {labels[i]}')
    axs[0].fill_between(x_axis, defs_fit_med[i] - defs_fit_std[i], defs_fit_med[i] + defs_fit_std[i], alpha=0.3, label='_nolegend_')

for i in range(len(atts_fit_med)):
    axs[1].plot(x_axis, atts_fit_med[i], linewidth=2, label=f'Step Size: {labels[i]}')
    axs[1].fill_between(x_axis, atts_fit_med[i] - atts_fit_std[i], atts_fit_med[i] + atts_fit_std[i], alpha=0.3, label='_nolegend_')


budget_gens = [(run_params["generations"]//num_budget)*i for i in range(0, num_budget)]
for i in range(1, len(budget_gens)):
    axs[0].axvline(x=budget_gens[i], color="black", linestyle='--')
    axs[1].axvline(x=budget_gens[i], color="black", linestyle='--')

trans0 = axs[0].get_xaxis_transform()
trans1 = axs[1].get_xaxis_transform()
for i, bgen in enumerate(budget_gens):
    axs[0].text(bgen + budget_gens[1] // 2, 0.94, f'{budgets[i]}', transform=trans0, ha='center', va='bottom', fontsize=FONTSIZE, fontweight="bold")
    axs[1].text(bgen + budget_gens[1] // 2, 0.94, f'{budgets[i]}', transform=trans1, ha='center', va='bottom', fontsize=FONTSIZE, fontweight="bold")

axs[0].set_xlabel('Generation')
axs[1].set_xlabel('Generation')
# ax.set_xticks([])
axs[0].set_ylabel('Avg. Med. Payoff')
axs[1].set_ylabel('Avg. Med. Payoff')
axs[0].set_title("Fixed Mutation Step Size: Defender")
axs[1].set_title("Fixed Mutation Step Size: Attacker") 
# axs[0].legend()
axs[1].legend()
axs[0].grid(True, alpha=0.3)
axs[1].grid(True, alpha=0.3)

# plt.tight_layout()
plt.savefig(plot_dir_path + "fixed_step_subplots.png", bbox_inches='tight')
plt.show()